# Gộp batch và lịch sử thành kết quả tích lũy theo ngày

Chạy lần lượt 3 cell trên Google Colab. Sửa `BATCH_FOLDER` nếu cần. Folder được chia sẻ cần có shortcut trong My Drive.

Nguồn dữ liệu: tất cả `worker_*_batch_*.jsonl` ở folder gốc + `results/DD-MM-YYYY/success.jsonl` và `errors.jsonl` của hôm qua + hai file của hôm nay nếu đã có. Ngày lấy theo giờ Việt Nam. Folder hôm qua phải có sẵn đủ hai file; notebook không tìm ngày thay thế.

Mỗi URL gốc chỉ giữ một bản ghi, so sánh sau khi bỏ khoảng trắng hai đầu. Success luôn được ưu tiên: URL có success sẽ không còn trong errors. Nếu cùng trạng thái, ưu tiên batch có thời gian sửa đổi mới hơn, rồi dữ liệu hôm nay, rồi hôm qua. Nội dung JSON được giữ nguyên; URL khác query vẫn là URL khác.

Kết quả được ghi vào hai file `.part` trước. Khi đọc và gộp hoàn tất, chúng thay `success.jsonl` và `errors.jsonl` của hôm nay. Vì vậy, chạy lần thứ hai trong ngày sẽ đọc cả kết quả lần đầu và tạo lại hai file, không nhân đôi URL. Không xóa batch gốc hoặc thay file hôm qua.

Notebook đọc từng dòng, lượt success trước rồi errors; RAM chỉ giữ tập URL, không giữ toàn bộ Markdown. JSON lỗi, URL thiếu hoặc `success` không phải boolean sẽ dừng gộp.

Chỉ chạy một notebook gộp tại một thời điểm. Chạy lại cell cấu hình nếu qua ngày mới. Các batch xuất hiện sau khi lập danh sách nguồn sẽ được lấy ở lần gộp sau. Notebook crawl vẫn chỉ đọc lịch sử hôm qua; file hôm nay này phục vụ skip từ ngày tiếp theo.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")


In [ ]:
from pathlib import Path
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

BATCH_FOLDER = Path("/content/drive/MyDrive/lucraftmeds_crawl")

local_date = datetime.now(ZoneInfo("Asia/Ho_Chi_Minh")).date()
TODAY = local_date.strftime("%d-%m-%Y")
YESTERDAY = (local_date - timedelta(days=1)).strftime("%d-%m-%Y")

RESULTS_FOLDER = BATCH_FOLDER / "results"
YESTERDAY_FOLDER = RESULTS_FOLDER / YESTERDAY
OUTPUT_FOLDER = RESULTS_FOLDER / TODAY

if not BATCH_FOLDER.is_dir():
    raise FileNotFoundError(f"Không tìm thấy folder: {BATCH_FOLDER}")
for name in ("success.jsonl", "errors.jsonl"):
    if not (YESTERDAY_FOLDER / name).is_file():
        raise FileNotFoundError(f"Thiếu dữ liệu hôm qua: {YESTERDAY_FOLDER / name}")

OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
print("Lịch sử hôm qua:", YESTERDAY_FOLDER)
print("Kết quả hôm nay:", OUTPUT_FOLDER)


In [ ]:
import json

# Batch mới hơn được ưu tiên nếu cùng URL và cùng trạng thái.
batch_files = sorted(
    (path for path in BATCH_FOLDER.glob("worker_*_batch_*.jsonl") if path.is_file()),
    key=lambda path: (path.stat().st_mtime_ns, path.name),
    reverse=True,
)

success_path = OUTPUT_FOLDER / "success.jsonl"
errors_path = OUTPUT_FOLDER / "errors.jsonl"
if success_path.exists() != errors_path.exists():
    raise ValueError("Folder hôm nay chỉ có một file kết quả; cần kiểm tra trước khi gộp.")

# Chỉ đọc file chính thức, không đọc các file .part.
sources = batch_files + [
    path
    for folder in (OUTPUT_FOLDER, YESTERDAY_FOLDER)
    for name in ("success.jsonl", "errors.jsonl")
    if (path := folder / name).is_file()
]

def read_records(path):
    with path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, 1):
            if not line.strip():
                continue
            try:
                record = json.loads(line)
            except ValueError as exc:
                raise ValueError(f"JSON lỗi: {path}, dòng {line_number}") from exc
            if not isinstance(record, dict) or not isinstance(record.get("success"), bool):
                raise ValueError(f"Thiếu success true/false: {path}, dòng {line_number}")
            url = record.get("url")
            if not isinstance(url, str) or not url.strip():
                raise ValueError(f"Thiếu URL: {path}, dòng {line_number}")
            yield url.strip(), record["success"], line.rstrip("\r\n") + "\n"

success_temp = OUTPUT_FOLDER / "success.jsonl.part"
errors_temp = OUTPUT_FOLDER / "errors.jsonl.part"
seen_urls = set()
counts = {True: 0, False: 0}

try:
    # Ghi success trước: mọi error của URL đã thành công đều bị loại.
    for ok, temp_path in ((True, success_temp), (False, errors_temp)):
        with temp_path.open("w", encoding="utf-8") as output:
            for source in sources:
                for url, success, line in read_records(source):
                    if success == ok and url not in seen_urls:
                        output.write(line)
                        seen_urls.add(url)
                        counts[ok] += 1

    # Kết quả hôm nay cũ đã được đọc xong trước khi thay bằng bản mới.
    success_temp.replace(success_path)
    errors_temp.replace(errors_path)
finally:
    success_temp.unlink(missing_ok=True)
    errors_temp.unlink(missing_ok=True)

print(f"Đã gộp {len(batch_files)} batch và lịch sử từ {YESTERDAY}, {TODAY}.")
print(f"Tổng URL duy nhất: {len(seen_urls):,}")
print(f"Success: {counts[True]:,} dòng → {success_path}")
print(f"Errors:  {counts[False]:,} dòng → {errors_path}")
